In [1]:
#B
import pandas as pd
orders = pd.DataFrame({
    'OrderID':[101, 102, 103, 104, 105, 106, 107, 108],
    'UnitPrice':[12.5, 45.0, 8.99, 120.0, 19.99, 60.0, 5.5, 89.0],
    'Quantity':[3, 1, 10, 2, 5, 1, 20, 4],
    'CustomerSince':pd.to_datetime(['2021-02-10','2022-11-05','2020-07-19',
                                      '2023-01-30','2019-09-12','2022-04-23',
                                      '2021-12-01','2023-06-15']),
    'OrderDate':pd.to_datetime(['2023-09-01','2023-09-03','2023-09-05',
                                      '2023-09-08','2023-09-10','2023-09-12',
                                      '2023-09-14','2023-09-18']),
    'Country':['India','UK','India','USA','UK','India','USA','India'],
    'Returned':[0, 0, 1, 0, 1, 0, 1, 0],
})

In [2]:
#B1
#a
orders['Revenue']=orders['UnitPrice']*orders['Quantity']
print(orders['Revenue'])

#b
mean=orders.groupby('Country')['UnitPrice'].transform('mean')
orders['Price_to_CountryAvg']=orders['UnitPrice']/mean
print(orders[['Country','Price_to_CountryAvg']])

#a is arithmetic and b is ratio feature

0     37.50
1     45.00
2     89.90
3    240.00
4     99.95
5     60.00
6    110.00
7    356.00
Name: Revenue, dtype: float64
  Country  Price_to_CountryAvg
0   India             0.293272
1      UK             1.384828
2   India             0.210921
3     USA             1.912351
4      UK             0.615172
5   India             1.407707
6     USA             0.087649
7   India             2.088099


In [3]:
#B2
from sklearn.preprocessing import PolynomialFeatures
poly=PolynomialFeatures(degree=2,interaction_only=True,include_bias=False)
#interaction_only=only keeps interactive terms
#include_bias=true
X=orders[['UnitPrice','Quantity']]
X_poly=poly.fit_transform(X)
poly_cols=poly.get_feature_names_out(['UnitPrice','Quantity'])
df_poly = pd.DataFrame(X_poly, columns=poly_cols) 
# print(df_poly)
print(pd.concat([orders['Revenue'],df_poly],axis=1))

# Hand-engineering Revenue explicitly is more interpretable and makes the business meaning clear. 
# Using PolynomialFeatures may create many unnecessary interaction columns and increase complexity.


   Revenue  UnitPrice  Quantity  UnitPrice Quantity
0    37.50      12.50       3.0               37.50
1    45.00      45.00       1.0               45.00
2    89.90       8.99      10.0               89.90
3   240.00     120.00       2.0              240.00
4    99.95      19.99       5.0               99.95
5    60.00      60.00       1.0               60.00
6   110.00       5.50      20.0              110.00
7   356.00      89.00       4.0              356.00


In [4]:
#B3
import numpy as np

orders['Order_Mnth']=orders['OrderDate'].dt.month
orders['Order_DayOfWeek']=orders['OrderDate'].dt.dayofweek
orders['Order_IsWeekend']=orders['OrderDate'].dt.dayofweek.isin([5,6]).astype(int)
orders['Customer_Tenure_Days']=(orders['OrderDate']-orders['CustomerSince']).dt.days

orders['Month_sin'] = np.sin(2 * np.pi*orders['Order_DayOfWeek']/7)
orders['Month_cos'] = np.cos(2 * np.pi*orders['Order_DayOfWeek']/7)

print(orders[['Order_Mnth','Order_DayOfWeek','Order_IsWeekend','Customer_Tenure_Days','Month_sin','Month_cos']])

   Order_Mnth  Order_DayOfWeek  Order_IsWeekend  Customer_Tenure_Days  \
0           9                4                0                   933   
1           9                6                1                   302   
2           9                1                0                  1143   
3           9                4                0                   221   
4           9                6                1                  1459   
5           9                1                0                   507   
6           9                3                0                   652   
7           9                0                0                    95   

   Month_sin  Month_cos  
0  -0.433884  -0.900969  
1  -0.781831   0.623490  
2   0.781831   0.623490  
3  -0.433884  -0.900969  
4  -0.781831   0.623490  
5   0.781831   0.623490  
6   0.433884  -0.900969  
7   0.000000   1.000000  


In [5]:
#B4
from sklearn.feature_selection import VarianceThreshold

orders['Currency']='INR'
orders.loc[orders.index[-1],'Curreny']='USD'

orders=pd.get_dummies(orders,columns=['Currency'],drop_first=False)

numeric_cols=orders.select_dtypes(include='number')
selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_cols)
kept=numeric_cols.columns[selector.get_support()].tolist()
#getsupport returns  boolean value
dropped = numeric_cols.columns[~selector.get_support()].tolist()

print("Kept:", kept)
print("Dropped:", dropped)

# A near-constant column is a column whose values are almost the same for all rows, with only a few exceptions.
# It has very low variance, meaning the values hardly change.

corr_features = orders[
[
'Revenue',
'UnitPrice',
'Quantity',
'Price_to_CountryAvg',
'Customer_Tenure_Days'
]
]
corr_matrix = corr_features.corr().abs()
print(corr_matrix)
threshold = 0.90
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

to_drop = [
    col
    for col in upper_tri.columns
    if any(upper_tri[col] > threshold)
]

print(to_drop)
target_corr = orders[
[
'Revenue',
'UnitPrice',
'Quantity',
'Price_to_CountryAvg',
'Customer_Tenure_Days',
'Returned'
]
].corr()['Returned']

print(target_corr)

Kept: ['OrderID', 'UnitPrice', 'Quantity', 'Returned', 'Revenue', 'Price_to_CountryAvg', 'Order_DayOfWeek', 'Order_IsWeekend', 'Customer_Tenure_Days', 'Month_sin', 'Month_cos']
Dropped: ['Order_Mnth']
                       Revenue  UnitPrice  Quantity  Price_to_CountryAvg  \
Revenue               1.000000   0.707493  0.049762             0.664038   
UnitPrice             0.707493   1.000000  0.547258             0.938893   
Quantity              0.049762   0.547258  1.000000             0.645716   
Price_to_CountryAvg   0.664038   0.938893  0.645716             1.000000   
Customer_Tenure_Days  0.543998   0.752692  0.260878             0.782260   

                      Customer_Tenure_Days  
Revenue                           0.543998  
UnitPrice                         0.752692  
Quantity                          0.260878  
Price_to_CountryAvg               0.782260  
Customer_Tenure_Days              1.000000  
['Price_to_CountryAvg']
Revenue                -0.221970
UnitPrice      

In [7]:
#B5
X = orders[
[
'Revenue',
'UnitPrice',
'Quantity',
'Price_to_CountryAvg',
'Customer_Tenure_Days',
'Month_sin',
'Month_cos',
'Order_IsWeekend'
]
]
y = orders['Returned']

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

selector_k = SelectKBest(
    score_func=f_classif,k=4
)
selector_k.fit(X, y)
scores = pd.Series(
    selector_k.scores_,
    index=X.columns
).sort_values(
    ascending=False
)
print(scores)

selected_kbest = X.columns[
    selector_k.get_support()
].tolist()

print(selected_kbest)

# b
# No
# chi2 requires all feature values to be non-negative.
# Features such as DayOfWeek_sin DayOfWeek_cos
# contain negative values.
# Therefore chi2 cannot be used directly.
# First apply-----MinMaxScaler
# and then use

Quantity                8.165767
Customer_Tenure_Days    6.703542
Price_to_CountryAvg     6.594874
UnitPrice               4.722980
Month_sin               0.407282
Revenue                 0.310945
Order_IsWeekend         0.136364
Month_cos               0.001588
dtype: float64
['UnitPrice', 'Quantity', 'Price_to_CountryAvg', 'Customer_Tenure_Days']


In [ ]:
#B6
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(
    max_iter=500
)

rfe = RFE(
    estimator=model,
    n_features_to_select=4,
    step=1
)
rfe.fit(X_scaled, y)

rankings = pd.Series(
    rfe.ranking_,
    index=X.columns
).sort_values()
print(rankings)

selected_rfe = X.columns[
    rfe.support_
].tolist()
print(selected_rfe)

#b
#Whether the features are identical depends entirely on your dataset, but they frequently differ.SelectKBest is a univariate filter
# method. It evaluates each feature independently of the others, scoring them using statistical tests (e.g., ANOVA F-value or mutual 
# information). It simply ranks them and picks the top \(k\).RFE (Recursive Feature Elimination) is a multivariate wrapper method. It 
# evaluates features in combination with one another by training a specific machine learning model (e.g., Random Forest, SVM). It 
# recursively removes the weakest features and rebuilds the model, capturing how features interact together.

#Handling Redundancy: SelectKBest might select two features that are highly correlated with the target variable, even if they contain 
# the exact same information. RFE removes redundant features because it evaluates the marginal contribution of a feature to the 
# model's accuracy when paired with the remaining features.

# Feature Interactions: Univariate methods fail to capture non-linear interactions where two individually weak features become
# highly predictive when used together. RFE preserves these interactive features if they collectively improve the model's performance.

# Model Specificity: RFE's selection is strictly tied to the inductive biases and assumptions of the underlying estimator you use to 
# evaluate it, whereas SelectKBest relies purely on general statistical distribution relative to the target

UnitPrice               1
Quantity                1
Price_to_CountryAvg     1
Customer_Tenure_Days    1
Month_cos               2
Order_IsWeekend         3
Month_sin               4
Revenue                 5
dtype: int64
['UnitPrice', 'Quantity', 'Price_to_CountryAvg', 'Customer_Tenure_Days']
